<p><font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Multimodal - Bild
</b></font> </br></p>

---

<p><font color='darkblue' size="4">
ℹ️ <b>Hinweis</b>
</font></p>

Dieses Notebook verwendet die direkte OpenAI-API für Bildgenerierung, Bildbearbeitung und Bildanalyse. Dadurch bleiben die Beispiele kompakt und ohne zusätzliche Wrapper-Schicht.

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

from genai_lib.model_config import IMAGE_GENERATION
from genai_lib.utilities import (
    check_environment,
    copy_from_github,
    get_ipinfo,
    install_packages,
    mermaid,
    mprint,
    setup_api_keys,
)


setup_api_keys(["OPENAI_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }

install_packages(["ultralytics"])

In [ ]:
#@title 📂 Dokumente, Bilder { display-mode: "form" }

import shutil

shutil.rmtree("files", ignore_errors=True)
copy_from_github(
    source="ralf-42/GenAI/02_daten/02_bild",
    target="files",
    mask="apfel.png",
)
copy_from_github(
    source="ralf-42/GenAI/02_daten/02_bild",
    target="files",
    mask="peoples.png",
)
copy_from_github(
    source="ralf-42/GenAI/02_daten/02_bild",
    target="files",
    mask="hedra_cyborg*.png",
)

In [ ]:
#@markdown 🛠️ Hilfsfunktionen { display-mode: "form" }

import base64
from io import BytesIO

from IPython.display import Image as IPImage, display
from openai import OpenAI
from PIL import Image as PILImage, ImageDraw
from ultralytics import YOLO

IMAGE_DIR = "/content/files"
VISION_MODEL = "gpt-5.4-mini"
OUTPAINT_SIZE = (1536, 1024)
client = OpenAI()


def encode_image(image_path):
    """Kodiert eine lokale Bilddatei als Base64-String."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def show_img(image, width=512):
    """Zeigt ein PIL-Bild skaliert an."""
    ratio = width / image.width
    new_size = (width, int(image.height * ratio))
    display(image.resize(new_size))


def image_from_response(response):
    """Baut aus der OpenAI-Antwort ein PIL-Bild."""
    img_data = base64.b64decode(response.data[0].b64_json)
    return PILImage.open(BytesIO(img_data))


def analyze_image(image_path, prompt, model=VISION_MODEL):
    """Analysiert ein Bild mit einem Vision-Modell."""
    base64_image = encode_image(image_path)
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
        ]
    }]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    return response.choices[0].message.content


def build_outpainting_assets(image_path, target_size=OUTPAINT_SIZE):
    """Erzeugt Canvas und Maske für Outpainting."""
    orig = PILImage.open(image_path).convert("RGBA")
    canvas = PILImage.new("RGBA", target_size, (0, 0, 0, 0))
    x_offset = (target_size[0] - orig.width) // 2
    y_offset = (target_size[1] - orig.height) // 2
    canvas.paste(orig, (x_offset, y_offset))

    mask = PILImage.new("RGBA", target_size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(mask)
    draw.rectangle(
        [x_offset, y_offset, x_offset + orig.width, y_offset + orig.height],
        fill=(255, 255, 255, 255),
    )
    return orig, canvas, mask

# 1 | Überblick
---

In diesem Notebook stehen vier Aufgaben im Mittelpunkt: Bildgenerierung, Bildanalyse, Bildbearbeitung und Objekterkennung mit YOLO. Die ersten drei Teile nutzen die OpenAI-API direkt, YOLO bleibt als eigener klassischer CV-Abschnitt erhalten.

| Abschnitt | Inhalt |
|---|---|
| Bildgenerierung | Text zu Bild mit GPT Image 2 |
| Bildanalyse | Klassifikation und Beschreibung mit `gpt-5.4-mini` |
| Bildbearbeitung | Inpainting und Outpainting |
| YOLO | Objekterkennung mit Bounding Boxes |

In [ ]:
#@markdown   <p><font size="4" color='green'>  Überblick</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    A[Prompt oder Bild] --> B[OpenAI API]
    B --> C[Bildgenerierung
GPT Image 2]
    B --> D[Bildanalyse
gpt-5.4-mini]
    B --> E[Bildbearbeitung
GPT Image 2]
    A --> F[YOLO]
    F --> G[Objekterkennung]
"""

mermaid(diagram, width=900)

<p><font color='black' size="5">
Übersicht Bildverarbeitung
</font></p>

| **Kategorie**                        | **Aufgabe**                      | **Beschreibung**                                                                      |
| ------------------------------------ | -------------------------------- | ------------------------------------------------------------------------------------- |
| 🧠 **Analyse & Klassifikation**      | **Bildklassifikation**           | Zuordnung eines Bildes zu vordefinierten Klassen (z. B. Katze, Hund, Auto).           |
|                                      | **Objekterkennung**                  | Erkennung und Lokalisierung mehrerer Objekte in einem Bild mit Bounding Boxes.        |
|                                      | Bildsegmentierung                | Pixelgenaue Zuordnung von Bildbereichen zu Klassen (z. B. Straße, Baum, Mensch).      |
|                                      | Gesichtserkennung                | Identifikation oder Verifikation von Personen auf Bildern.                            |
|                                      | Emotionserkennung                | Analyse der Gesichtsausdrücke zur Einschätzung von Emotionen.                         |
|                                      | Anomalieerkennung                | Erkennung ungewöhnlicher oder fehlerhafter Bildinhalte (z. B. Produktionsfehler).     |
| 📝 **Generierung & Transformation**  | **Bildgenerierung**              | Erzeugung synthetischer Bilder, z. B. aus Text (Text-to-Image) oder Rauschen.         |
|                                      | Bild-zu-Bild-Übersetzung         | Umwandlung von Bildern (z. B. Skizze → Foto, Tag → Nacht).                            |
|                                      | Stiltransfer                     | Übertragung des Stils eines Bildes auf ein anderes (z. B. Van Gogh → Foto).           |
|                                      | Super-Resolution                 | Hochskalierung von Bildern mit künstlicher Detailschärfung.                           |
|                                      | Bildrestauration                 | Entfernung von Rauschen, Kratzern oder Artefakten in alten oder beschädigten Bildern. |
|                                      | Colorisierung                    | Umfärben von Schwarzweiß-Bildern mit realistischen Farben.                            |
| 🧩 **Ergänzung & Vervollständigung** | **Inpainting**                       | Auffüllen fehlender oder beschädigter Bildbereiche.                                   |
|                                      | **Bildvervollständigung**        | Vorhersage fehlender Bildbereiche basierend auf Kontext.                              |
|                                      | Hintergrundentfernung            | Trennung des Vordergrundobjekts vom Hintergrund.                                      |
| 📚 **Informationsgewinnung**         | Texterkennung (OCR)              | Erkennung und Extraktion von Text aus Bildern.                                        |
|                                      | Schrifterkennung                 | Identifikation von Schriftarten oder Handschriften.                                   |
|                                      | **Bildbeschreibung (Captioning)**    | Automatische Erzeugung eines beschreibenden Textes zu einem Bild.                     |
|                                      | visuelle Fragebeantwortung (VQA) | Beantwortung von Fragen basierend auf einem Bildinhalt.                               |
| 🛡️ **Sicherheit & Strukturierung**  | Deepfake-Erkennung               | Analyse, ob ein Bild manipuliert oder synthetisch erzeugt wurde.                      |
|                                      | Gesichtsverpixelung              | Anonymisierung durch Unkenntlichmachung von Gesichtern.                               |
|                                      | Wasserzeichen-Erkennung          | Detektion sichtbarer oder unsichtbarer Wasserzeichen.                                 |
|                                      | Formatkonvertierung              | Umwandlung zwischen Bildformaten (z. B. PNG ↔ JPG).                                   |

# 2 | Bildgenerierung
---

Die Bildgenerierung mit Künstlicher Intelligenz (KI) ermöglicht es, aus einfachen Texteingaben oder bestehenden Bildern völlig neue, realistische oder kreative Grafiken zu erstellen. Moderne KI-Modelle wie GPT-Image setzen dabei auf fortschrittliche Algorithmen, die Texte präzise interpretieren und in beeindruckende visuelle Darstellungen umsetzen. Diese Technologie findet Anwendung in vielen Bereichen – von Design und Marketing über Kunst bis hin zur Bildung – und eröffnet neue kreative Möglichkeiten für Einsteiger und Profis gleichermaßen.


<p><font color='darkblue' size="4">
💡 <b>Viz</b>
</font></p>


- [Merkmals-Filter](https://editor.p5js.org/ralf.bendig.rb/full/zLXqi5u6f)
- [Merkmals-Filter-Anwendung](https://editor.p5js.org/ralf.bendig.rb/full/Xi2uabjR9)
- [CNN Explainer](https://poloclub.github.io/cnn-explainer/)


<p><font color='black' size="5">
Prompt-Beispiele
</font></p>

Prompts aus: [Versäumte Bilder](https://bilderinstitut.de/versaeumte-bilder-bonn)

**Foto Elvira Fölzer** standing proud as a female archaeologist, looking directly in the camera, as a 70 year old, on an bexcavation side with antic shards and ruins, photography from 1920, leica style, film gain --s 250 --v 6.1


**Foto Maria von Linden** standing proud with crossed arms, as a 70 year old female scientist, dressed like a man, with slicked back hair, looking like a man, wearing a lab cout, standing in a lecture hall in Bonn in front of students, teaching chemistry, leica style from 1920 --v 6.0 --s 250


**Foto Leah Goldberg** as a 30 year old female scientist, sitting proud behind a desk with books, writing, illuminated by a small lamp, leica style from 1933, film gain --s 250 --v 6.1

... oder einfach: Erstelle eine Bild von einem Prosche 911 auf einer Küstenstraße. Das Bild sollte foto-realistisch und schwarz-weiss sein.

GPT Image 2 erzeugt Bilder direkt aus einem Prompt. Für das Notebook reicht ein klarer Prompt mit Stil, Motiv und gewünschter Wirkung.

In [ ]:
#@markdown   <p><font size="4" color='green'>  GPT Image 2</font> </br></p>

prompt = "Portrait of Elvira Fölzer, a 70-year-old female archaeologist standing proudly on an excavation site with ancient shards and ruins, photographed in 1920s Leica style, film grain, high detail."

response = client.images.generate(model=IMAGE_GENERATION, prompt=prompt, size="1024x1024", quality="high", n=1)
img = image_from_response(response)
img.save("bild_gpt_image_2.png")
show_img(img)

# 3 | Bildanalyse
---

Die Bildklassifizierung ist eine grundlegende Aufgabe in der Computer Vision, bei der einem gesamten Bild eine einzige Kategorie (Klasse) zugewiesen wird.

**Wie funktioniert Bildklassifizierung?**
+ Eingabe: Ein einzelnes Bild wird als Eingabe verwendet.
+ Merkmalsextraktion: Ein neuronales Netzwerk analysiert das Bild und extrahiert relevante Merkmale (z. B. Kanten, Farben, Formen, Muster).
+ Klassifizierung: Das Modell ordnet das Bild einer vordefinierten Kategorie zu, z. B. "Hund", "Katze" oder "Auto".
+ Ausgabe: Eine einzige Klasse (Label) mit einer Wahrscheinlichkeitsbewertung wird zurückgegeben.


Beispiel für Bildklassifizierung
Stelle dir vor, du hast ein Bild von einem Hund. Ein Bildklassifizierungsmodell verarbeitet das Bild und gibt die Kategorie "Hund" mit einer bestimmten Wahrscheinlichkeit (z. B. 95 %) zurück.

+ Eingabebild: Ein Bild eines Hundes
+ Modell-Ausgabe: "Hund" (95%)

Falls das Bild eine Katze zeigt, gibt das Modell möglicherweise "Katze" (90%) als Ergebnis zurück.

**Einschränkungen der Bildklassifizierung**     
Das Modell kann nur eine Klasse pro Bild erkennen, auch wenn mehrere Objekte im Bild vorhanden sind.
Es gibt keine Information über die Position oder Anzahl der Objekte im Bild.
Anwendungsfälle für Bildklassifizierung
+ Erkennung von medizinischen Anomalien (z. B. Klassifikation von Röntgenbildern)
+ Identifikation von Pflanzen oder Tieren anhand von Bildern
+ Sentiment-Analyse anhand von Gesichtsmerkmalen

**Bekannte Modelle für Bildklassifizierung**    
+ CNNs (Convolutional Neural Networks) wie ResNet, VGG, EfficientNet
+ Pretrained Modelle: MobileNet, Inception, AlexNet

Das Modell **google/vit-base-patch16-224** ist ein Vision Transformer (ViT) Modell, das von Google entwickelt wurde und sich auf Bildklassifizierungsaufgaben spezialisiert hat. Es gehört zur Familie der Transformer-Modelle, die ursprünglich für die Verarbeitung natürlicher Sprache (NLP) konzipiert wurden, aber hier auf visuelle Daten angewendet werden.

Bildklassifikation und Bildbeschreibung laufen hier über dieselbe Funktion. Der Unterschied steckt nur im Prompt.

In [ ]:
image_path = f"{IMAGE_DIR}/apfel.png"
display(IPImage(image_path, width=500))

result = analyze_image(image_path, "Was ist auf diesem Bild abgebildet? Nenne nur das wesentliche Objekt.")
mprint(f"**Klassifikation:** {result}")

In [ ]:
description_image_path = f"{IMAGE_DIR}/peoples.png"
display(IPImage(description_image_path, width=750))

result = analyze_image(description_image_path, "Beschreibe das Bild in 2-3 Sätzen.")
mprint(f"**Bildbeschreibung:** \n\n{result}")

# 4 | Bildbearbeitung
---

Inpainting ist eine Technik im Bereich der künstlichen Intelligenz und Bildverarbeitung, bei der bestimmte Teile eines Bildes automatisch ergänzt oder repariert werden. Die Technik wird eingesetzt, um:

+ Beschädigte Bereiche in Bildern zu rekonstruieren
+ Unerwünschte Objekte aus Bildern zu entfernen
+ Fehlende Teile in Bildern zu ergänzen

**Inpainting mit Maske:**   
Bei Inpainting mit Masken geht es darum, genau zu definieren, welche Bereiche eines Bildes rekonstruiert werden sollen. Die Maske ist dabei ein Schlüsselelement, das dem Algorithmus mitteilt, wo er arbeiten soll. Eine Maske ist ein Binärbild (Schwarz-Weiß-Bild), das die gleichen Dimensionen wie das Originalbild hat:

+ Weiße Bereiche (255): Zeigen an, welche Teile des Bildes rekonstruiert/gefüllt werden sollen
+ Schwarze Bereiche (0): Zeigen an, welche Teile des Originalbildes erhalten bleiben sollen

**Inpainting** verändert nur den markierten Bereich. Outpainting erweitert das Bild über den ursprünglichen Rand hinaus. Beide Beispiele verwenden dieselbe OpenAI-API, nur mit anderer Vorbereitung.

In [ ]:
prompt = "Ersetze den Kopf durch den Kopf eines alten Mannes. Behalte Licht, Schatten und Stil des Originals."
image_path = f"{IMAGE_DIR}/hedra_cyborg.png"
mask_path = f"{IMAGE_DIR}/hedra_cyborg_mask_image.png"

with open(image_path, "rb") as img_f, open(mask_path, "rb") as mask_f:
    response = client.images.edit(
        model=IMAGE_GENERATION,
        image=img_f,
        mask=mask_f,
        prompt=prompt,
        size="1024x1024",
        output_format="png",
    )

img = image_from_response(response)
show_img(img)

**Outpainting** ist eine Technik der Bildbearbeitung und KI-gestützten Bildgenerierung, bei der ein vorhandenes Bild über seine ursprünglichen Grenzen hinaus erweitert wird. Dabei werden neue Bildinhalte an den Rändern ergänzt, die stilistisch und thematisch zum Original passen. Outpainting wird häufig verwendet, um Bilder zu vergrößern, Hintergründe zu erweitern oder kreative Kompositionen zu schaffen. Moderne Methoden basieren oft auf neuronalen Netzwerken wie Stable Diffusion, die mithilfe von Textbeschreibungen (Prompts) realistische und kohärente Bildbereiche hinzufügen können. So entsteht ein nahtlos erweitertes Bild, das über das ursprüngliche Motiv hinausgeht.

Beim **Outpainting** bleibt der ursprüngliche Bildkern erhalten. Die Maske markiert den Bereich, der ergänzt werden soll.

In [ ]:
orig, canvas, mask = build_outpainting_assets(f"{IMAGE_DIR}/hedra_cyborg.png")
canvas.save("canvas.png")
mask.save("mask.png")

show_img(orig, width=512)

prompt = "Erweitere das Bild nach außen. Füge harmonisch passende Umgebung, Licht und Texturen hinzu. Behalte den Stil des Originals."

with open("canvas.png", "rb") as img, open("mask.png", "rb") as m:
    response = client.images.edit(
        model=IMAGE_GENERATION,
        image=img,
        mask=m,
        prompt=prompt,
        size="1536x1024",
        quality="high",
        output_format="png",
    )

img = image_from_response(response)
img.save("outpaint_1536x1024.png")
show_img(img, width=768)

# 5 | Objekterkennung
---

Die Objekterkennung geht einen Schritt weiter als die Bildklassifizierung. Hier wird nicht nur bestimmt, welche Objekte in einem Bild vorhanden sind, sondern auch wo sie sich befinden.

**Wie funktioniert Objekterkennung?**   
+ Eingabe: Ein einzelnes Bild wird als Eingabe verwendet.
+ Merkmalsextraktion & Vorschläge für Regionen: Das Modell analysiert das Bild und identifiziert potenzielle Bereiche, in denen sich Objekte befinden könnten.
+ Klassifizierung & Lokalisierung: Jedes erkannte Objekt wird einer bestimmten Klasse zugeordnet, und die Position wird durch eine Bounding Box (ein rechteckiger Bereich um das Objekt) beschrieben.
+ Ausgabe: Eine Liste mit allen erkannten Objekten, ihrer Klasse und ihrer Position im Bild wird zurückgegeben.


**Beispiel für Objekterkennung**    
Stelle dir vor, du hast ein Bild mit einem Hund, einer Katze und einem Auto. Ein Objekterkennungsmodell kann alle drei Objekte gleichzeitig identifizieren und ihre Positionen angeben.

+ Eingabebild: Ein Bild mit einem Hund, einer Katze und einem Auto
+ Modell-Ausgabe:
>Hund (95%) - Position: (x1, y1, x2, y2)   
Katze (90%) - Position: (x3, y3, x4, y4)   
Auto (99%) - Position: (x5, y5, x6, y6)   

Hierbei sind (x1, y1, x2, y2) die Koordinaten der Bounding Box für den Hund, (x3, y3, x4, y4) für die Katze usw.

**Einschränkungen der Objekterkennung**   
Aufwendigere Berechnungen im Vergleich zur einfachen Bildklassifizierung.
Schwieriger zu trainieren, da genaue Positionen für Trainingsdaten erforderlich sind.
Anwendungsfälle für Objekterkennung
+ Autonomes Fahren (Erkennung von Fußgängern, Verkehrsschildern)
+ Überwachungssysteme (Erkennen von Eindringlingen oder gefährlichen Objekten)
+ Analyse von Bildern im Einzelhandel (z. B. automatische Produktzählung in Regalen)

**Bekannte Modelle für Objekterkennung**   
+ YOLO (You Only Look Once) – sehr schnell und effizient
+ Faster R-CNN – präzise, aber langsamer
+ SSD (Single Shot MultiBox Detector) – guter Kompromiss zwischen Geschwindigkeit und Genauigkeit

<p><font color='black' size="5">
YOLO
</font></p>

**YOLO (You Only Look Once)** ist ein leistungsfähiger Algorithmus für die Objekterkennung in Bildern und Videos mithilfe künstlicher Intelligenz (KI). Es handelt sich um ein Modell, das auf Convolutional Neural Networks (CNN) basiert und sich durch seine Geschwindigkeit und Genauigkeit auszeichnet.

**Funktionsweise**

YOLO teilt ein Bild in ein Gitter auf und verarbeitet es in einem einzigen Durchlauf. Jede Gitterzelle ist für die Erkennung von Objekten in ihrem Bereich verantwortlich. Der Algorithmus sagt gleichzeitig Begrenzungsrahmen (Bounding Boxes) und Klassenwahrscheinlichkeiten für Objekte voraus.

**Vorteile**

1. **Geschwindigkeit**: YOLO kann Bilder in Echtzeit mit bis zu 45 Bildern pro Sekunde verarbeiten, was es ideal für Anwendungen wie Videoüberwachung oder autonomes Fahren macht.

2. **Genauigkeit**: Trotz seiner Geschwindigkeit erreicht YOLO eine hohe Erkennungsgenauigkeit und weist nur wenige Hintergrundfehler auf.

3. **Verallgemeinerungsfähigkeit**: Neuere Versionen von YOLO bieten eine verbesserte Leistung bei der Erkennung von Objekten in neuen Umgebungen.

**Entwicklung**

Seit seiner Einführung im Jahr 2015 hat YOLO mehrere Iterationen durchlaufen, wobei jede Version Verbesserungen und Optimierungen mit sich brachte. Die neueste Version ist YOLO v7, die weitere Fortschritte in Bezug auf Geschwindigkeit und Genauigkeit bietet.

YOLO hat die Objekterkennung revolutioniert und findet Anwendung in verschiedenen Bereichen wie autonomen Fahrzeugen, Überwachungssystemen, Robotik und generativer KI.

- [Ultralytics | YOLO](https://www.ultralytics.com/de)

Der Code verwendet die Ultralytics YOLOv8 Bibliothek, um Objekterkennung auf einem hochgeladenen Bild durchzuführen. Er lädt ein vortrainiertes YOLOv8m-Modell, ermöglicht das Hochladen eines Bildes über Google Colab, führt die Objekterkennung auf diesem Bild durch (mit einer Konfidenzschwelle von 0.2), speichert das Bild mit den Erkennungsergebnissen und die entsprechenden Textdateien und zeigt schließlich das bearbeitete Bild in der Colab-Ausgabe an.

In [ ]:
yolo_model = YOLO("yolov8m.pt")
detection_image_path = f"{IMAGE_DIR}/peoples.png"

In [ ]:
results = yolo_model.predict(source=detection_image_path, conf=0.2)
annotated = PILImage.fromarray(results[0].plot()[:, :, ::-1])

mprint("### Objekterkennung")
mprint("---")
show_img(annotated, width=750)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei einer Aufgabe, bei der man festhängt, kann zum Beispiel Gemini in Google Colab helfen: Fehlermeldungen besser verstehen, Ideen für Teilschritte entwickeln oder Code-Varianten prüfen.
> Wichtig ist: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Erstelle mit GPT Image 2 ein Coverbild für das Buch. Nutze Buchtitel und Synopsis als Grundlage und achte darauf, dass das Bild den Inhalt sichtbar transportiert.

**Erledigt wenn:** Das generierte Bild passt visuell zum Buchtitel und zur Synopsis aus M05.

In [ ]:
# Starter-Template: Coverbild mit GPT Image 2 generieren

buchtitel = "..."   # aus M05
synopsis = "..."    # aus M05, 2–3 Sätze

bild_prompt = f"Buchcover für '{buchtitel}': {synopsis}. Realistisch, detailreich."

# response = client.images.generate(model=IMAGE_GENERATION, prompt=bild_prompt, size='1024x1024')
# img = image_from_response(response)
# show_img(img)

**Aufbau**

Erstelle ein Outpainting zu einem selbst gewählten Bild.

**Vertiefung**

Erstelle ein Inpainting zu einem selbst gewählten Bild.

[Inpaint Mask Maker](https://huggingface.co/spaces/r3gm/inpaint-mask-maker)

[Inpaint Mask Maker](https://huggingface.co/spaces/stevhliu/inpaint-mask-maker)

# B | Dokumente zum Weiterlesen
---

📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Multimodal: Bild](https://ralf-42.github.io/GenAI/09-multimodal/multimodal-bild.html)
- [Embeddings](https://ralf-42.github.io/GenAI/03-grundlagen/embeddings.html)